# Case study: date, interaction and target-encoded store features

*Session 9, block 1 practice. Author: course team, licence CC-BY-4.0.*

We add three groups of features to the review sample and measure, with a time-respecting cross-validation, whether each group improves macro-F1:

1. **dates**: year and cyclical month,
2. **an interaction**: negations × text length,
3. **store**: rare stores grouped, then target-encoded with cross-fitting.

All feature construction happens inside a scikit-learn `Pipeline`, so it is refitted in every training fold. Theory: [01-dates-interactions-and-categories.md](../theory/01-dates-interactions-and-categories.md).

Run from the course environment (`uv run jupyter lab` in the repository root). The data must have been prepared with `case-study/prepare_data.py`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# find the repository root, wherever the notebook is started from
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study" / "data").exists())
DATA = ROOT / "case-study" / "data"

reviews = pd.read_parquet(DATA / "train_sample.parquet")
products = pd.read_parquet(DATA / "products.parquet", columns=["parent_asin", "store"])
df = (reviews.merge(products, on="parent_asin", how="left")
             .sort_values("date").reset_index(drop=True))          # time order for TimeSeriesSplit
df["store"] = df["store"].fillna("missing")
print(df.shape)
df["label"].value_counts(normalize=True).round(3)

## 1. Text statistics as the base features

The same seven statistics as on theory page 2, plus three metadata columns.

In [ ]:
NEGATIONS = r"\b(?:not|no|never|don't|doesn't|didn't|isn't|wasn't|won't|can't)\b"


def text_stats(frame):
    """Seven numeric statistics of review text and title (Session 9, page 2)."""
    text, title = frame["text"].fillna(""), frame["title"].fillna("")
    n_chars = text.str.len()
    return pd.DataFrame({
        "log_chars": np.log1p(n_chars),
        "n_words": text.str.split().str.len(),
        "n_exclaim": text.str.count("!"),
        "n_question": text.str.count(r"\?"),
        "upper_share": text.str.count(r"[A-Z]") / n_chars.clip(lower=1),
        "n_negations": text.str.lower().str.count(NEGATIONS),
        "title_words": title.str.split().str.len(),
    }, index=frame.index)

base = pd.concat([text_stats(df), df[["verified_purchase", "helpful_vote", "n_images"]].astype(float)], axis=1)
base.describe().round(2)

## 2. Date features with a custom transformer

`date_features` needs nothing from the training data, so a `FunctionTransformer` is enough.

**Task 2a.** Look at the share of each label per month and per year. Which of the two do you expect to help the model?

In [ ]:
from sklearn.preprocessing import FunctionTransformer


def date_features(X):
    month = X["date"].dt.month
    return pd.DataFrame({"year": X["date"].dt.year,
                         "month_sin": np.sin(2 * np.pi * month / 12),
                         "month_cos": np.cos(2 * np.pi * month / 12)}, index=X.index)


dates = FunctionTransformer(date_features, feature_names_out=lambda tf, names: ["year", "month_sin", "month_cos"])
print(dates.fit_transform(df[["date"]]).head(3).round(2))

# Task 2a: label shares per year and per month
pd.crosstab(df["date"].dt.year, df["label"], normalize="index").tail(8).round(3)

In [ ]:
# TODO (Task 2a): the same table per month (df["date"].dt.month)


## 3. An interaction

A long text with many negations is probably a complaint; a short text with one "not" may not be. The product `n_negations × log_chars` lets an additive model express this.

**Task 3.** Propose a second interaction, add it to `interactions` below and check the score in Section 5.

In [ ]:
def interactions(X):
    return pd.DataFrame({"neg_x_len": X["n_negations"] * X["log_chars"]}, index=X.index)
    # TODO (Task 3): add a second product, e.g. "exclaim_x_upper": X["n_exclaim"] * X["upper_share"]


inter = FunctionTransformer(interactions, feature_names_out=lambda tf, names: ["neg_x_len"])
inter.fit_transform(base).describe().round(2)

## 4. Store: rare grouping and target encoding

`RareGrouper` must learn the list of frequent stores in `fit`, so it is a small class. `TargetEncoder` then replaces each store by three smoothed class shares (`store_neg`, `store_neu`, `store_pos`), cross-fitted inside `fit_transform`.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import TargetEncoder


class RareGrouper(BaseEstimator, TransformerMixin):
    """Replace categories seen fewer than min_count times in the training data by 'other'."""

    def __init__(self, min_count=20):
        self.min_count = min_count

    def fit(self, X, y=None):
        counts = X.iloc[:, 0].value_counts()
        self.frequent_ = set(counts[counts >= self.min_count].index)
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        return self

    def transform(self, X):
        col = X.iloc[:, 0]
        return col.where(col.isin(self.frequent_), "other").to_frame()

    def get_feature_names_out(self, input_features=None):
        return self.feature_names_in_                 # one output column per input column


store = make_pipeline(RareGrouper(min_count=20),
                      TargetEncoder(target_type="multiclass", cv=KFold(5, shuffle=True, random_state=0)))
encoded = store.fit_transform(df[["store"]], df["label"])
print(encoded.shape, store.get_feature_names_out())
print(RareGrouper(20).fit(df[["store"]]).transform(df[["store"]]).value_counts().head(5))

## 5. Compare the feature groups

A `TimeSeriesSplit` on the date-sorted sample always validates on reviews that are later than the training reviews, like the leaderboard. The model is a histogram gradient boosting classifier with balanced class weights (Session 10 explains it in detail).

**Task 5.** Which group helps most? Is the gain of the store encoding larger than the fold-to-fold standard deviation?

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

X = pd.concat([base, df[["date", "store"]]], axis=1)
y = df["label"]
text_cols = list(base.columns)
cv = TimeSeriesSplit(n_splits=5)


def make_model(use_dates=False, use_inter=False, use_store=False):
    parts = [("base", "passthrough", text_cols)]
    if use_dates:
        parts.append(("dates", dates, ["date"]))
    if use_inter:
        parts.append(("inter", inter, ["n_negations", "log_chars"]))
    if use_store:
        parts.append(("store", store, ["store"]))
    return make_pipeline(ColumnTransformer(parts),
                         HistGradientBoostingClassifier(class_weight="balanced", random_state=0))


configs = {"text + metadata": {}, "+ dates": {"use_dates": True}, "+ interaction": {"use_inter": True},
           "+ store": {"use_store": True},
           "all": {"use_dates": True, "use_inter": True, "use_store": True}}
results = {}
for name, kw in configs.items():
    s = cross_val_score(make_model(**kw), X, y, cv=cv, scoring="f1_macro")
    results[name] = {"mean": s.mean(), "std": s.std()}
pd.DataFrame(results).T.round(3)

## 6. Check the leakage of a naive encoding

**Task 6.** Replace the `TargetEncoder` by a naive group mean computed on all rows (for example `df.groupby("store")["label"]...`) and evaluate it in the same cross-validation. Compare with Section 5 and explain the difference in two sentences.

**Task 7 (optional).** Tune `min_count` of `RareGrouper` over `[5, 20, 100]` with `GridSearchCV` (parameter name `columntransformer__store__raregrouper__min_count`).

In [ ]:
# TODO (Task 6 and 7)
